In [25]:
import pandas as pd
from scipy.stats import kendalltau,weightedtau
import numpy as np
import matplotlib.pyplot as plt
import dcor
import seaborn as sns

## Feature Size

In [26]:
IF_9_1 = pd.read_csv("ClassImbalance_Diamonds/IF_9_1_class.csv")
IF_8_2 = pd.read_csv("ClassImbalance_Diamonds/IF_8_2_class.csv")
IF_7_3 = pd.read_csv("ClassImbalance_Diamonds/IF_7_3_class.csv")
IF_6_4 = pd.read_csv("ClassImbalance_Diamonds/IF_6_4_class.csv")
IF_5_5 = pd.read_csv("ClassImbalance_Diamonds/IF_5_5_class.csv")

TC_9_1 = pd.read_csv("ClassImbalance_Diamonds/TC_9_1_class.csv")
TC_8_2 = pd.read_csv("ClassImbalance_Diamonds/TC_8_2_class.csv")
TC_7_3 = pd.read_csv("ClassImbalance_Diamonds/TC_7_3_class.csv")
TC_6_4 = pd.read_csv("ClassImbalance_Diamonds/TC_6_4_class.csv")
TC_5_5 = pd.read_csv("ClassImbalance_Diamonds/TC_5_5_class.csv")

Label_1 = pd.read_csv("ClassImbalance_Diamonds/9_1_class_labelIDs.csv")
Label_2 = pd.read_csv("ClassImbalance_Diamonds/8_2_class_labelIDs.csv")
Label_3 = pd.read_csv("ClassImbalance_Diamonds/7_3_class_labelIDs.csv")
Label_4 = pd.read_csv("ClassImbalance_Diamonds/6_4_class_labelIDs.csv")
Label_5 = pd.read_csv("ClassImbalance_Diamonds/5_5_class_labelIDs.csv")

In [27]:
sorted_IF_lists = [IF_9_1, IF_8_2,IF_7_3,IF_6_4,IF_5_5]
sorted_TC_lists = [TC_9_1, TC_8_2,TC_7_3,TC_6_4,TC_5_5]
ratio=[(9,1),(8,2),(7,3),(6,4),(5,5)]
sorted_label_lists = [Label_1,Label_2,Label_3,Label_4,Label_5]
num_lists = len(sorted_IF_lists)
random_selection_rate = [0.1, 0.2, 0.3, 0.4, 0.5]

In [28]:
K = 200

In [29]:
def compute_sign_aware_lift(
    influence_df,
    label_df,
    minority_rate,
    k=100,
    score_col="Score"
):
    """
    Compute minority lift in:
      1. top-k signed influence scores,
      2. bottom-k signed influence scores,
      3. top-k absolute influence scores.

    Lift = minority proportion in selected set
           -----------------------------------
           minority proportion in full training set
    """

    ranked = influence_df.merge(
        label_df,
        left_on="Train_ID",
        right_on="id",
        how="left",
        validate="one_to_one"
    )

    ranked = ranked.drop(columns=["id"])

    if ranked["label"].isna().any():
        missing = ranked["label"].isna().sum()
        raise ValueError(f"{missing} influence rows have no matching label.")

    if score_col not in ranked.columns:
        raise KeyError(
            f"Cannot find score column '{score_col}'. "
            f"Available columns: {ranked.columns.tolist()}"
        )

    if k > len(ranked):
        raise ValueError(
            f"k={k} exceeds the number of training samples ({len(ranked)})."
        )

    # Highest signed influence
    top_signed = ranked.nlargest(k, score_col)

    # Lowest, most negative signed influence
    bottom_signed = ranked.nsmallest(k, score_col)

    # Largest influence magnitude
    top_absolute = (
        ranked.assign(Abs_Score=ranked[score_col].abs())
        .nlargest(k, "Abs_Score")
    )

    top_rate = top_signed["label"].mean()
    bottom_rate = bottom_signed["label"].mean()
    abs_top_rate = top_absolute["label"].mean()

    return {
        "Lift@Top": top_rate / minority_rate,
        "Lift@Bottom": bottom_rate / minority_rate,
        "Lift@AbsTop": abs_top_rate / minority_rate,
        "Minority@Top": top_rate,
        "Minority@Bottom": bottom_rate,
        "Minority@AbsTop": abs_top_rate
    }

In [30]:
records = []

for i in range(num_lists):
    minority_rate = float(random_selection_rate[i])

    # FOIF
    if_result = compute_sign_aware_lift(
        influence_df=sorted_IF_lists[i],
        label_df=sorted_label_lists[i],
        minority_rate=minority_rate,
        k=K,
        score_col="Score"
    )

    records.append({
        "Dataset": r"Diamonds",
        "Method": "FOIF",
        "Ratio": f"{ratio[i][1]}:{ratio[i][0]}",
        "k": K,
        "Lift@Top": if_result["Lift@Top"],
        "Lift@Bottom": if_result["Lift@Bottom"],
        "Lift@AbsTop": if_result["Lift@AbsTop"]
    })

    # TracIn
    tc_result = compute_sign_aware_lift(
        influence_df=sorted_TC_lists[i],
        label_df=sorted_label_lists[i],
        minority_rate=minority_rate,
        k=K,
        score_col="Score"
    )

    records.append({
        "Dataset": r"Diamonds",
        "Method": "TracIn",
        "Ratio": f"{ratio[i][1]}:{ratio[i][0]}",
        "k": K,
        "Lift@Top": tc_result["Lift@Top"],
        "Lift@Bottom": tc_result["Lift@Bottom"],
        "Lift@AbsTop": tc_result["Lift@AbsTop"]
    })

sign_aware_lift_table = pd.DataFrame(records)

print(sign_aware_lift_table.round(3))

    Dataset  Method Ratio    k  Lift@Top  Lift@Bottom  Lift@AbsTop
0  Diamonds    FOIF   1:9  200     9.950        0.850        9.000
1  Diamonds  TracIn   1:9  200    10.000        0.000        6.850
2  Diamonds    FOIF   2:8  200     5.000        2.375        4.675
3  Diamonds  TracIn   2:8  200     5.000        0.450        4.475
4  Diamonds    FOIF   3:7  200     3.333        1.917        3.050
5  Diamonds  TracIn   3:7  200     3.333        0.533        2.900
6  Diamonds    FOIF   4:6  200     2.300        1.262        2.075
7  Diamonds  TracIn   4:6  200     2.500        0.650        1.938
8  Diamonds    FOIF   5:5  200     0.820        0.830        0.830
9  Diamonds  TracIn   5:5  200     0.000        1.960        0.670


In [31]:
table_1_9 = sign_aware_lift_table[
    sign_aware_lift_table["Ratio"] == "1:9"
].copy()

table_1_9 = table_1_9[
    [
        "Dataset",
        "Method",
        "Ratio",
        "Lift@Top",
        "Lift@Bottom",
        "Lift@AbsTop"
    ]
]

print(table_1_9.round(3))

    Dataset  Method Ratio  Lift@Top  Lift@Bottom  Lift@AbsTop
0  Diamonds    FOIF   1:9      9.95         0.85         9.00
1  Diamonds  TracIn   1:9     10.00         0.00         6.85
